<a href="https://colab.research.google.com/github/nestarShaxzod/IFRS-9-Expected-Credit-Loss-ECL-Automation-Eng/blob/main/ifrs_9_ecl_automatization_eng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***IFRS 9 Expected Credit Loss (ECL) Automation***
------------------------------------------------------------------------
**Objective:** Demonstrate the automation of Expected Credit Loss (ECL) calculation in accordance with IFRS 9 requirements using Python (Pandas) and DuckDB SQL.


Data and source file paths have been anonymized for public release.




In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import duckdb
pd.set_option('display.float_format', '{:,.2f}'.format)

# ***Input and Output Data Paths***

In [ ]:
path_input = r'./data/input'
path_output = r'./data/output'

# ***Data Type Conversion Functions***

In [ ]:
#______________________________________________Function for Converting Data to Date Type______________________________________________
def def_date_type(df, col):
    df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d.%m.%Y', errors = 'coerce')
    return df

#______________________________________________Function for Numeric Data Type Conversion________________________________________________
def def_numb_type(df, col):
    df[col] = df[col].str.replace(',', '.', regex = True)
    df[col] = df[col].str.replace(r'\s+', '', regex = True)
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

# ***Loading and Merging Source Files***

In [ ]:
file_path = glob.glob(os.path.join(path_input, '*.txt'))
all_frame = []

for f in file_path:

    line_skip_row = 0
    with open(f, 'r', encoding = 'utf-8') as f_open:                             # For correct data import, remove all rows preceding the header row containing the keyword "Period"
        for numb, row_content in enumerate(f_open):
          if 'Период' in row_content:
            line_skip_row = numb
            break

    df = pd.read_csv(f, sep = '\t', skiprows = line_skip_row, encoding = 'utf-8', usecols=[0,2,3,4,5,7,8]) # Select only the required columns
    all_frame.append(df)

concat_df = pd.concat(all_frame, ignore_index=True)

concat_df = def_date_type(concat_df, 'Период')
concat_df = def_numb_type(concat_df, 'Unnamed: 5')
concat_df = def_numb_type(concat_df, 'Unnamed: 8')

concat_df = concat_df.rename(columns={'Период': 'Date', 'Аналитика Дт': 'Аналитика_Дт', 'Аналитика Кт': 'Аналитика_Кт', 'Unnamed: 5': 'Сумма_Дт', 'Unnamed: 8': 'Сумма_Кт'})
concat_df.head(7)

pd.concat([concat_df.head(5), concat_df.tail(5)])                                # Display the resulting dataset

# ***Data Processing and Transformation Using DuckDB SQL***

In [ ]:
duckdb.register("duck_ecl_calcul", concat_df)

query_df = """
      SELECT
        *,
        CASE
            WHEN "Дебет" LIKE '40%' THEN STRING_SPLIT(REPLACE(Аналитика_Дт, CHR(13), ''), CHR(10))[1]
            ELSE STRING_SPLIT(REPLACE(Аналитика_Кт, CHR(13), ''), CHR(10))[1]
        END AS "Counterparty",                                                  -- Extract counterparty information from transaction details
        CASE
            WHEN "Дебет" LIKE '40%' THEN STRING_SPLIT(REPLACE(Аналитика_Дт, CHR(13), ''), CHR(10))[2]
            ELSE STRING_SPLIT(REPLACE(Аналитика_Кт, CHR(13), ''), CHR(10))[2]
        END AS "Contract",                                                      -- Extract contract information from transaction details
        CASE
            WHEN "Дебет" LIKE '40%' THEN "Сумма_Дт"
            ELSE 0
        END AS "Inflow",                                                        -- Treat debt increases as incoming transactions
        CASE
            WHEN "Кредит" LIKE '40%' THEN "Сумма_Кт" *-1
            ELSE 0
        END AS "Payment"                                                        -- Record debt settlements by counterparties as outgoing transactions (negative values)
      FROM duck_ecl_calcul
      WHERE "Date" is not null and "Date" <= '2025-12-31'
      ORDER BY "Date", "Counterparty", "Contract"                               -- Filter transactions by date within each counterparty and contract group
"""

duck_df = duckdb.query(query_df).df()



pd.concat([duck_df.head(5),duck_df.tail(5)])                         # Display the resulting dataset

# ***Additional Data Processing Using DuckDB SQL***

In [ ]:
query_df2 = """
      WITH T1 AS (
          SELECT
            "Date",
            "Counterparty",
            "Contract",
            "Inflow",
            "Payment",
            SUM("Inflow") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date") AS "Cum_Inflow", -- Calculate cumulative receipts
            SUM("Payment") OVER(PARTITION BY "Counterparty", "Contract") AS "Total_Payment",             -- Calculate cumulative repayments by contract
        FROM duck_df
      ),
          T2 AS (
            SELECT
              *,
              CASE
                WHEN "Cum_Inflow" < ABS("Total_Payment") THEN 0
                ELSE "Cum_Inflow" + "Total_Payment"
              END AS "Cum_Unpaid_amount"
            FROM T1
          )
                SELECT
                  *,
                  CASE
                      WHEN COALESCE(LAG("Cum_Unpaid_amount") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date"),0) = "Cum_Unpaid_amount" THEN 0
                      ELSE "Cum_Unpaid_amount" - LAG("Cum_Unpaid_amount") OVER(PARTITION BY "Counterparty", "Contract" ORDER BY "Date")
                  END AS "Unpaid_amount"
                FROM T2

"""

duck_source_info= duckdb.query(query_df2).df()

pd.concat([duck_source_info.head(5), duck_source_info.tail(5)])           # Display the resulting dataset

# ***Aging Analysis Calculation and Aging Bucket Classification***

In [ ]:
date_input = '2025-12-31'

query_df3 = f"""

          SELECT
              Date,
              "Counterparty",
              "Contract",
              "Unpaid_amount",
              DATEDIFF('day', "Date", '{date_input}') AS "Days_to_date",                -- Calculate the difference between the reporting date and the receivable origination date
              CASE
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 30 THEN '0-30'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 60 THEN '31-60'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 90 THEN '61-90'
                  WHEN DATEDIFF('day', "Date", '{date_input}') <= 120 THEN '91-120'
                  ELSE '120 и более'                                                    -- Assign overdue days to the corresponding aging buckets
              END AS "Aging_Bucket"
          FROM duck_source_info
          WHERE "Unpaid_amount" > 1                                                     -- Filter the required data
          ORDER BY "Date", "Counterparty", "Contract"

"""

duck_df_31_12_2025 = duckdb.sql(query_df3).df()

pd.concat([duck_df_31_12_2025.head(5),duck_df_31_12_2025.tail(5)])                  # Display the resulting dataset

# ***PD and LGD Parameter Calculation***

In [ ]:
# Apply predefined PD and LGD values to simplify the calculation logic and improve code readability.

df_pd = {'0-30': 0.05,
          '31-60': 0.15,
          '61-90': 0.25,
          '91-120': 0.35,
          '120 и более': 1
          }

df_lgd = 0.45

# ***Expected Credit Loss (ECL) Calculation***

In [ ]:
df_ecl = duck_df_31_12_2025.copy()

df_ecl['PD'] = df_ecl['Aging_Bucket'].map(df_pd)                                # Map aging buckets to the corresponding PD values
df_ecl['LGD'] = df_lgd
df_ecl['ECL'] = df_ecl['Unpaid_amount'] * df_ecl['PD'] * df_ecl['LGD']          # Calculate the total Expected Credit Loss (ECL) in accordance with IFRS 9

pd.concat([df_ecl.head(5),df_ecl.tail(5)])                                      # Display the resulting dataset

# ***Saving the Processing Results***

In [ ]:
path_save = os.path.join(path_output, 'ECL.xlsx')

with pd.ExcelWriter(path_save) as writer:
  df_ecl.to_excel(writer, sheet_name='ECL', index=False)
